# 4.0 Serve and Evaluate the Compressed VLM

In this notebook you will learn how to:

- Serve a pruned, distilled (and optionally quantized) VLM with [vLLM](https://docs.vllm.ai/).
- Evaluate the served VLM on RealWorldQA with [`lmms-eval`](https://github.com/EvolvingLMMs-Lab/lmms-eval).

Prerequisites: notebooks 01-03 have produced one or more VLM checkpoints at, for example:

- `/workspace/user_homes/lmikaelyan/Qwen3-VL-8B-Instruct-pruned-34-32-10752` — pruned
- `/workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-smoke` — pruned + distilled
- `/workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-fp8` — pruned + distilled + FP8

Pick whichever checkpoint you want to evaluate.

---
## 4.1 Background

vLLM hosts the VLM behind an OpenAI-compatible HTTP API. `lmms-eval` then talks to that API the same way it would talk to GPT-4V, batching benchmark samples and computing the task metric. This decouples the model implementation (HF + vLLM kernels) from the benchmark harness, and means the same evaluation code runs against BF16, FP8, NVFP4, or any served model.

We use **RealWorldQA** as the smoke benchmark — 765 single-image, single-answer questions about real-world scenes. Cheap to run (a few minutes on H100) and discriminative enough to tell pruned-only and pruned + distilled checkpoints apart.

---
## 4.2 Serve the VLM with vLLM

Start vLLM in a separate terminal (or with `&` to background it from here). Once the server is up, the OpenAI-compatible API is reachable at `http://localhost:8000/v1`.

```bash
vllm serve /workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-smoke \
    --trust-remote-code --dtype bfloat16
```

For an FP8 export the command is identical — vLLM autodetects the FP8 quantization from the checkpoint's `config.json`:

```bash
vllm serve /workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-fp8 \
    --trust-remote-code --dtype bfloat16
```

Optionally constrain image preprocessing to keep VRAM bounded:

```bash
vllm serve /workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-smoke \
    --trust-remote-code --dtype bfloat16 \
    --mm-processor-kwargs '{"max_pixels": 2097152, "min_pixels": 262144, "max_num_frames": 1}'
```

Wait for the line `Application startup complete.` in the vLLM logs before proceeding. The cell below polls the health endpoint for you.

In [ ]:
import requests, time
for _ in range(120):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print("vLLM is ready"); break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    print("vLLM not reachable after 10 min — check the serve logs")

Quick smoke test of the served endpoint with one image:

In [ ]:
import base64, requests, json
from PIL import Image
from io import BytesIO

MODEL_PATH = "/workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-smoke"
img_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg"
img = Image.open(BytesIO(requests.get(img_url, timeout=10).content)).convert("RGB")
buf = BytesIO(); img.save(buf, format="PNG"); b64 = base64.b64encode(buf.getvalue()).decode()

resp = requests.post(
    "http://localhost:8000/v1/chat/completions",
    headers={"Authorization": "Bearer token-abc123"},
    json={
        "model": MODEL_PATH,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": "Describe the image in one short sentence."},
            ],
        }],
        "max_tokens": 64, "temperature": 0,
    }, timeout=120,
)
print(resp.json()["choices"][0]["message"]["content"])

---
## 4.3 Evaluate with lmms-eval (RealWorldQA)

`lmms-eval` talks to vLLM through the OpenAI-compatible API. RealWorldQA is small (~765 samples), so this finishes in a few minutes.

The `--model openai --model_args model=<path>,base_url=http://localhost:8000/v1` form points lmms-eval at our local vLLM endpoint. The `OPENAI_API_KEY` is required by the OpenAI client but vLLM doesn't validate it — any token works.

In [ ]:
MODEL_PATH = "/workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-smoke"
LOG_DIR    = "/workspace/user_homes/lmikaelyan/logs/qwen3-vl-distilled-smoke-rwqa"

!OPENAI_API_KEY=token-abc123 python -m lmms_eval \
    --model openai \
    --model_args model={MODEL_PATH},base_url=http://localhost:8000/v1 \
    --tasks realworldqa \
    --batch_size 64 \
    --output_path {LOG_DIR}

Expected: a table per task with `exact_match` (and `stderr`). For Qwen3-VL-8B-Instruct, the unpruned BF16 baseline lands around 0.69 on RealWorldQA; a pruned-only model usually drops to ~0.60-0.65, and a pruned + distilled model recovers most of the gap.

Compare two checkpoints side-by-side by re-running the cell with a different `MODEL_PATH` (e.g. swap the distilled-smoke path for the FP8 path).

---
## 4.4 Shut down the server

When you're done, stop the `vllm serve` process you started in §4.2 (Ctrl-C in its terminal, or `pkill -f 'vllm serve'`).